In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
import MEArec as mr
from tabnanny import verbose
import spikeinterface as si
import numpy as np
from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components
import pickle
import pandas as pd
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model,
    build_sliding_cliques,
    calibration_model,
    SimpleAutoSort
)
import torch


In [2]:
recording, sorting = se.read_mearec("/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s.h5")
probe = recording.get_probe()
recording_recorded = spre.bandpass_filter(recording, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")
probe.set_contact_ids(recording.channel_ids)

In [3]:
output_folder = '/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s'
cliques = build_sliding_cliques(
    probe,
    clique_size=49,
    min_size=25,
    min_overlap=18,
    target_groups=12,
)

clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'clique_size': 49,
        'min_size': 25,
        'min_overlap': 18,
        'target_groups': 12,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
}

clique_info_path = f'{output_folder}/clique_info.pkl'
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

[INFO] Built 12 cliques (target 12)
       Clique 00: channels 192-12 (49 channels)
       Clique 01: channels 103-115 (49 channels)
       Clique 02: channels 303-123 (49 channels)
       Clique 03: channels 23-227 (49 channels)
       Clique 04: channels 31-43 (49 channels)
       Clique 05: channels 326-338 (49 channels)
       Clique 06: channels 334-154 (49 channels)
       Clique 07: channels 54-258 (49 channels)
       Clique 08: channels 254-74 (49 channels)
       Clique 09: channels 165-177 (49 channels)
       Clique 10: channels 173-185 (49 channels)
       Clique 11: channels 371-383 (49 channels)


In [4]:
# 设置基本参数（这些参数应该与生成数据的脚本保持一致）
segment_duration_seconds = 600  # 每段600秒
n_segments = 6  # 总共6段

# 获取recording的采样率
sampling_frequency = recording_f.get_sampling_frequency()
total_num_samples = recording_f.get_num_samples()

# 计算每段的采样点数
segment_num_samples = int(segment_duration_seconds * sampling_frequency)

# 计算每个segment的采样点范围（用于提取recording片段）
segment_sample_ranges = {}  # {segment_idx: (start_sample, end_sample)}
for seg_idx in range(n_segments):
    start_sample = seg_idx * segment_num_samples
    # 最后一段可能不足600s，使用实际结束位置
    if seg_idx == n_segments - 1:
        end_sample = total_num_samples
    else:
        end_sample = (seg_idx + 1) * segment_num_samples
    
    segment_sample_ranges[seg_idx] = (start_sample, end_sample)

# 设置输出文件夹
combined_output_base = output_folder

# ============================================================
# 测试模型：对每个clique、每个segment使用segment_0训练的5个模型进行验证
# ============================================================
print(f"\n{'='*60}")
print(f"开始测试模型：每个clique、每个segment使用segment_0训练的5个模型进行验证")
print(f"{'='*60}\n")

# 训练segment（使用segment_0训练的模型）
train_segment_idx = 0

# 对每个clique进行测试
for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"测试 Clique {clique_id}")
    print(f"{'='*60}")
    
    # 读取训练segment（segment_0）的neuron_inf
    train_segment_folder = f'{combined_output_base}/clique_{clique_id}/segment_{train_segment_idx}'
    train_neuron_inf_path = f'{train_segment_folder}/neuron_inf.pickle'
    
    if not os.path.exists(train_neuron_inf_path):
        print(f"  警告: {train_neuron_inf_path} 不存在，跳过")
        continue
    
    # 加载训练neuron_inf
    with open(train_neuron_inf_path, 'rb') as f:
        train_neuron_inf_dict = pickle.load(f)
    train_neuron_inf = neuron_inf_dict_to_dataframe(train_neuron_inf_dict)
    
    print(f"  训练Segment {train_segment_idx}: {len(train_neuron_inf)} 个神经元")
    
    # 对每个测试segment进行测试
    for test_segment_idx in range(n_segments):
        print(f"\n  {'='*60}")
        print(f"  测试 Segment {test_segment_idx}")
        print(f"  {'='*60}")
        
        # 读取测试segment的数据
        test_segment_folder = f'{combined_output_base}/clique_{clique_id}/segment_{test_segment_idx}'
        test_neuron_inf_path = f'{test_segment_folder}/neuron_inf.pickle'
        test_gt_detect_array_path = f'{test_segment_folder}/gt_detect_array.csv'
        
        if not os.path.exists(test_neuron_inf_path) or not os.path.exists(test_gt_detect_array_path):
            print(f"    警告: {test_segment_folder} 下没有找到数据文件，跳过")
            continue
        
        # 加载测试数据
        with open(test_neuron_inf_path, 'rb') as f:
            test_neuron_inf_dict = pickle.load(f)
        test_gt_detect_array = pd.read_csv(test_gt_detect_array_path)
        
        test_neuron_inf = neuron_inf_dict_to_dataframe(test_neuron_inf_dict)
        
        print(f"    测试Segment {test_segment_idx} 数据:")
        print(f"      - Neurons: {len(test_neuron_inf)}")
        print(f"      - Spikes: {len(test_gt_detect_array)}")
        
        if len(test_neuron_inf) == 0 or len(test_gt_detect_array) == 0:
            print(f"    警告: Segment {test_segment_idx} 没有数据，跳过")
            continue
        
        # 比较训练和测试的neuron，找出重合的神经元
        train_neuron_ids = set(train_neuron_inf['Neuron'].unique())
        test_neuron_ids = set(test_neuron_inf['Neuron'].unique())
        matched_neuron_ids = train_neuron_ids & test_neuron_ids
        
        print(f"    重合的神经元: {len(matched_neuron_ids)} (在segment_{train_segment_idx}和segment_{test_segment_idx}中都存在)")
        
        # 为test_neuron_inf添加neuron_match列（用于calibration_model的评估）
        test_neuron_inf_matched = test_neuron_inf.copy()
        test_neuron_inf_matched['neuron_match'] = test_neuron_inf_matched['Neuron'].apply(
            lambda x: x if x in matched_neuron_ids else 'unmatch'
        )
        
        # 获取测试segment对应的recording片段
        segment_start_sample, segment_end_sample = segment_sample_ranges[test_segment_idx]
        test_recording_segment = recording_f.frame_slice(
            start_frame=segment_start_sample,
            end_frame=segment_end_sample
        )
        
        # 获取recording_clique
        test_recording_clique = get_recording_clique(test_recording_segment, clique)
        print(f"    测试recording通道数: {len(test_recording_clique.get_channel_ids())}")
        
        # 重复实验5次，每次使用不同的模型权重
        n_repeats = 5
        n_channels = test_recording_clique.get_num_channels()
        samplepoints = 30  # left_sample + right_sample = 10 + 20
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        for repeat_idx in range(1, n_repeats + 1):
            print(f"\n    ===== 重复实验 {repeat_idx}/{n_repeats} (使用 model_{repeat_idx}) =====")
            
            model_save_dir = f'{train_segment_folder}/model_{repeat_idx}'
            noise_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_noise_clsfier.pth'
            label_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_label_clsfier.pth'
            
            # 检查模型文件是否存在
            if not os.path.exists(noise_model_path) or not os.path.exists(label_model_path):
                print(f"      警告: model_{repeat_idx} 的权重文件不存在，跳过")
                continue
            
            # 加载keep_id列表
            keep_id_path = f'{model_save_dir}/keep_id.pkl'
            if not os.path.exists(keep_id_path):
                print(f"      警告: keep_id.pkl不存在 {keep_id_path}，跳过")
                continue
            
            with open(keep_id_path, 'rb') as f:
                keep_id_list = pickle.load(f)
            
            # 创建模型
            autosort_model = SimpleAutoSort(
                ch_num=n_channels,
                samplepoints=samplepoints,
                device=device,
                set_shank_id=keep_id_list,
                save_dir=model_save_dir,
                pos_weight_noise=None,
                pos_weight_label=None
            )
            
            autosort_model.clsfier_noise.load_state_dict(torch.load(noise_model_path, map_location=device))
            autosort_model.clsfier_label.load_state_dict(torch.load(label_model_path, map_location=device))
            autosort_model.eval()
            
            # 运行calibration
            calibration_results = calibration_model(
                recording_f=test_recording_clique,
                autosort_model=autosort_model,
                train_neuron_inf=train_neuron_inf,
                calibration_duration_seconds=segment_duration_seconds,  # 使用segment的时长（600秒）
                n_additional_clusters=5,
                detection_params={
                    'thr_min': 2.5,
                    'thr_max': 10,
                    'distance': 6,
                    'wlen': 5,
                    'prominence': 10,
                    'max_firing_channel': 8,
                },
                window_params={
                    'left_sample': 10,
                    'right_sample': 20,
                },
                position_threshold=10.0,
                waveform_similarity_threshold=0.95,
                eval_neuron_inf=test_neuron_inf_matched,
                gt_detect_array=test_gt_detect_array,
                device=device
            )
            
            # 保存结果
            output_path = f"{test_segment_folder}/calibration_model_{repeat_idx}.pkl"
            with open(output_path, 'wb') as f:
                pickle.dump(calibration_results, f)
            
            print(f"      重复实验 {repeat_idx}/{n_repeats} 完成，结果已保存到: {output_path}")
        
        print(f"    Segment {test_segment_idx} 所有重复实验完成!")
    
    print(f"  Clique {clique_id} 所有测试完成!")

print("\n所有测试完成！")



开始测试模型：每个clique、每个segment使用segment_0训练的5个模型进行验证


测试 Clique 0
  训练Segment 0: 14 个神经元

  测试 Segment 0
    测试Segment 0 数据:
      - Neurons: 14
      - Spikes: 37742
    重合的神经元: 14 (在segment_0和segment_0中都存在)
    测试recording通道数: 49

    ===== 重复实验 1/5 (使用 model_1) =====
Using old detection method: extremum_channels
Using 11 valid channels from train neuron extremum_channels
eval_neuron_inf筛选: 所有14个neuron都在valid_channels中
gt_detect_array筛选: 所有37742个spikes都属于valid neurons
Stage 1: Calibration (first 60 seconds)
Loading first 600 seconds of data...
Data shape: (49, 6000000)

### 2. Threshold detection
Number of detected spikes: 447343
去重: 移除了25072个spikes（保留幅值更大的channel上的spike）
去重前: 447343个spikes, 去重后: 422271个spikes
Number of detected spikes after deduplication: 422271
GT匹配统计: 36078/37742 GT spikes被检测到 (召回率: 0.9559)

### 3. Extract waveforms

### 4. Noise classifier filtering


Noise classification: 100%|██████████| 207/207 [00:01<00:00, 155.99it/s]


Number of spikes passing noise classifier: 38081
Noise classifier准确率: 0.9930 (419333/422270)
GT spike通过noise classifier比例: 0.9435 (35611/37742)

### 5. Per-channel K-means clustering and matching

### 6. Firing rate filtering

### 7. Summary
Matching results:
  - Total clusters: 110
  - Matched clusters: 77
  - Matched neurons: 12
  - Invalid neurons (firing rate < 0.5 Hz): 0
  - Spikes matched to neurons: 29165
  - Spikes marked as noise: 8916
  - Total spikes after noise classifier: 38081

### Classification Accuracy Calculation
  Total spikes analyzed: 38081
  Overall accuracy: 0.7210 (72.10%)
  Accuracy (excluding noise): 0.9783 (97.83%)


Extracting way3 features for all spikes: 100%|██████████| 207/207 [00:23<00:00,  8.88it/s]


      重复实验 1/5 完成，结果已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/clique_0/segment_0/calibration_model_1.pkl

    ===== 重复实验 2/5 (使用 model_2) =====
Using old detection method: extremum_channels
Using 11 valid channels from train neuron extremum_channels
eval_neuron_inf筛选: 所有14个neuron都在valid_channels中
gt_detect_array筛选: 所有37742个spikes都属于valid neurons
Stage 1: Calibration (first 60 seconds)
Loading first 600 seconds of data...
Data shape: (49, 6000000)

### 2. Threshold detection
Number of detected spikes: 447343
去重: 移除了25072个spikes（保留幅值更大的channel上的spike）
去重前: 447343个spikes, 去重后: 422271个spikes
Number of detected spikes after deduplication: 422271
GT匹配统计: 36078/37742 GT spikes被检测到 (召回率: 0.9559)

### 3. Extract waveforms

### 4. Noise classifier filtering


...

      重复实验 4/5 完成，结果已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/clique_11/segment_5/calibration_model_4.pkl

    ===== 重复实验 5/5 (使用 model_5) =====
Using old detection method: extremum_channels
Using 15 valid channels from train neuron extremum_channels
eval_neuron_inf筛选: 所有20个neuron都在valid_channels中
gt_detect_array筛选: 所有30235个spikes都属于valid neurons
Stage 1: Calibration (first 60 seconds)
Loading first 600 seconds of data...
Data shape: (49, 6000000)

### 2. Threshold detection
Number of detected spikes: 598900
去重: 移除了25680个spikes（保留幅值更大的channel上的spike）
去重前: 598900个spikes, 去重后: 573220个spikes
Number of detected spikes after deduplication: 573220
GT匹配统计: 29074/30235 GT spikes被检测到 (召回率: 0.9616)

### 3. Extract waveforms

### 4. Noise classifier filtering


Noise classification: 100%|██████████| 280/280 [00:01<00:00, 178.01it/s]


Number of spikes passing noise classifier: 32172
Noise classifier准确率: 0.9881 (566419/573219)
GT spike通过noise classifier比例: 0.9004 (27223/30235)

### 5. Per-channel K-means clustering and matching

### 6. Firing rate filtering
  Neuron 978: firing rate 0.2333 Hz < 0.3 Hz, marked as invalid
  Neuron 242: firing rate 0.1883 Hz < 0.3 Hz, marked as invalid
  Marked 4 clusters and 253 spikes as noise due to low firing rate

### 7. Summary
Matching results:
  - Total clusters: 150
  - Matched clusters: 99
  - Matched neurons: 17
  - Invalid neurons (firing rate < 0.5 Hz): 2
  - Spikes matched to neurons: 26702
  - Spikes marked as noise: 5470
  - Total spikes after noise classifier: 32172

### Classification Accuracy Calculation
  Total spikes analyzed: 32172
  Overall accuracy: 0.7082 (70.82%)
  Accuracy (excluding noise): 0.9564 (95.64%)


Extracting way3 features for all spikes: 100%|██████████| 280/280 [00:28<00:00,  9.71it/s]


      重复实验 5/5 完成，结果已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/clique_11/segment_5/calibration_model_5.pkl
    Segment 5 所有重复实验完成!
  Clique 11 所有测试完成!

所有测试完成！
